# Read MS csv

## import

In [1]:
import io
import pandas as pd
import numpy as np
from pathlib import Path
# import re  # for debugging

## read csv

In [10]:
# Read the CSV file
file_path = Path('./20260427v2.csv')  # this should be in the same folder as this script, or provide a complete filepath
assert file_path.is_file(), f"File not found: {file_path}. Please double check the path and filename."
with open(file_path, 'r') as f:
    lines = f.readlines()

# instrument filepath from metadata 
instrument_filepath = lines[0].strip().split(',', 1)[1]  # split on first comma only
print(f"Instrument path: {instrument_filepath}")

Instrument path: D:\MassHunter\Data\ZD\202604\20260422_TigE-v2\QuantResults\2026 04 24.batch.bin


## find sequence and data table lines in the csv

In [11]:
# find sequence and data tables by looking for "Data File" in the table header row
table_starts = []  # (name_line_index, header_line_index, table_name)
for i, line in enumerate(lines):
    if line.strip().startswith('Data File'):
        table_starts.append((i - 1, i, lines[i - 1].strip().strip(',')))  # name row is the line just before

# find table names and print for checking 
table_names = [table_name for _, _, table_name in table_starts]
print(f"Found {len(table_starts)} tables {table_names} at indices:")
max_width = max(len(name) for name in table_names) + 2
for name_idx, header_idx, table_name in table_starts:
    print(f"  {table_name:<{max_width}} : {header_idx}")

Found 4 tables ['Sequence Table', 'Methionine', '5dA', '5 MTA'] at indices:
  Sequence Table   : 4
  Methionine       : 34
  5dA              : 50
  5 MTA            : 66


## parse into dataframes and clean

In [12]:
tables = {}
# parse into however many tables 
for t, (name_idx, header_idx, name) in enumerate(table_starts):
    end_idx = table_starts[t + 1][0] if t + 1 < len(table_starts) else len(lines)  # find last line of each table 
    chunk = ''.join(lines[header_idx:end_idx])  # add the header line in 
    # debugging print statements 
    # print(f"--- {name} ---")
    # print(repr(chunk[:200]))  # show raw first 200 chars including escape characters
    # first_line = chunk.split('\n')[0]
    # print(f"{name}: {repr(first_line)}")
    # print()
    tables[name] = pd.read_csv(io.StringIO(chunk), header=0, skip_blank_lines=True).dropna(how='all')

# make a tables dictionary 
tables = {name: table for name, table in tables.items()}

# show head of each table to check 
for name, table in tables.items():
    print(f"--- {name} ---")
    display(table.head())
    print()
# tables['Sequence Table'].head()

--- Sequence Table ---


,Data File,Name,Type,Vial,Vol.,Level,Acq. Method File
0,01.d,01 water,Blank,NaN,3,NaN,C18_MRM_5dA Met 5MTA_Trp ISTD.m
1,02.d,02 water,Blank,NaN,3,NaN,C18_MRM_5dA Met 5MTA_Trp ISTD.m
2,03.d,03 std 10 1,Cal,NaN,3,1.0,C18_MRM_5dA Met 5MTA_Trp ISTD.m
3,04.d,04 std 25 1,Cal,NaN,3,2.0,C18_MRM_5dA Met 5MTA_Trp ISTD.m
4,05.d,05 std 75 1,Cal,NaN,3,3.0,C18_MRM_5dA Met 5MTA_Trp ISTD.m



--- Methionine ---


,Data File,Type,ISTD,RT,Resp.,ISTD Resp,RR,Final Conc.,Exp. Conc.,Accuracy
0,01.d,Blank,Trp,0.795,44.501,3837.915,0.0116,0.0000,NaN,NaN
1,02.d,Blank,Trp,0.795,26.875,758.170,0.0354,0.0000,NaN,NaN
2,03.d,Cal,Trp,0.795,549928.459,2851487.753,0.1929,11.3900,10.8,105.5
3,04.d,Cal,Trp,0.795,776306.874,3255501.365,0.2385,21.4141,25.0,85.7
4,05.d,Cal,Trp,0.795,1871660.733,2700113.710,0.6932,81.1444,75.0,108.2



--- 5dA ---


,Data File,Type,ISTD,RT,Resp.,ISTD Resp,RR,Final Conc.,Exp. Conc.,Accuracy
0,01.d,Blank,Trp,1.409,1.991710e+02,3837.915,0.0519,0.0000,NaN,NaN
1,02.d,Blank,Trp,1.014,2.366150e+02,758.170,0.3121,0.0000,NaN,NaN
2,03.d,Cal,Trp,1.241,2.332200e+06,2851487.753,0.8179,9.2884,10.0,92.9
3,04.d,Cal,Trp,1.229,5.140821e+06,3255501.365,1.5791,27.0404,25.0,108.2
4,05.d,Cal,Trp,1.217,1.214598e+07,2700113.710,4.4983,73.5516,75.0,98.1



--- 5 MTA ---


,Data File,Type,ISTD,RT,Resp.,ISTD Resp,RR,Final Conc.,Exp. Conc.,Accuracy
0,01.d,Blank,Trp,3.273,1.203893e+04,3837.915,3.1368,1.6487,NaN,NaN
1,02.d,Blank,Trp,3.333,1.656391e+03,758.170,2.1847,0.0000,NaN,NaN
2,03.d,Cal,Trp,2.831,1.145390e+07,2851487.753,4.0168,10.3151,10.0,103.2
3,04.d,Cal,Trp,2.819,1.747225e+07,3255501.365,5.3670,22.4425,25.0,89.8
4,05.d,Cal,Trp,2.819,3.744101e+07,2700113.710,13.8665,81.1773,75.0,108.2


## rename columns and pivot

In [13]:
# Split sequence and data table names for separate processing
sequence_table_name = table_names[0]
data_table_names = table_names[1:]

# Rename columns in data tables with their table name as prefix
skip_cols = ['Type', 'ISTD']
double_rename = False
for name, table in tables.items():
    if name != sequence_table_name:
        try:
            tables[name] = table.drop(columns=skip_cols)
        except KeyError:
            double_rename = True 
            pass
if double_rename:
    print("Note: you may have doubly renamed your data tables. If you see something like \"Methionine_Methionine_RT\" \n" 
          "in your column names, run the cell \"parse into dataframes and clean\" above and then this cell again.\n")

for table_name in data_table_names:
    tables[table_name].columns = [f'{table_name}_{col}' if col != 'Data File' else 'Data File' for col in tables[table_name].columns]
# to check column renaming: 
# print("Renamed columns in data tables:")
# for table_name in data_table_names:
#     print(f"  {table_name}: {tables[table_name].columns.tolist()}")
# print()

# Merge data tables into the sequence table, matching on the 'Data File' column
final_df = tables[sequence_table_name]
for name in data_table_names:
    final_df = final_df.merge(tables[name], on='Data File', how='left')

# Remove any remaining blank rows
final_df = final_df.dropna(how='all')

# print("Column names:")
# print(final_df.columns.tolist())
print(f"Sequence table rows: {len(tables[sequence_table_name])}")
print(f"Data tables rows: {[len(tables[name]) for name in data_table_names]}")
print(f"Final dataframe shape (rows, cols): {final_df.shape}")
if final_df.shape[0] != len(tables[sequence_table_name]):
    print("Warning: final dataframe row count does not match sequence table row count. Check for merge issues.")
else:
    print("Row count matches sequence table row count, looks good.")
print(f"Target compounds in table: {data_table_names}")
print(f"First few rows:")
display(final_df.head())
# print("Full dataframe:")
# display(final_df)

Sequence table rows: 14
Data tables rows: [14, 14, 14]
Final dataframe shape (rows, cols): (14, 28)
Row count matches sequence table row count, looks good.
Target compounds in table: ['Methionine', '5dA', '5 MTA']
First few rows:


,Data File,Name,Type,Vial,Vol.,Level,Acq. Method File,Methionine_RT,Methionine_Resp.,Methionine_ISTD Resp,...,5dA_Final Conc.,5dA_Exp. Conc.,5dA_Accuracy,5 MTA_RT,5 MTA_Resp.,5 MTA_ISTD Resp,5 MTA_RR,5 MTA_Final Conc.,5 MTA_Exp. Conc.,5 MTA_Accuracy
0,01.d,01 water,Blank,NaN,3,NaN,C18_MRM_5dA Met 5MTA_Trp ISTD.m,0.795,44.501,3837.915,...,0.0000,NaN,NaN,3.273,1.203893e+04,3837.915,3.1368,1.6487,NaN,NaN
1,02.d,02 water,Blank,NaN,3,NaN,C18_MRM_5dA Met 5MTA_Trp ISTD.m,0.795,26.875,758.170,...,0.0000,NaN,NaN,3.333,1.656391e+03,758.170,2.1847,0.0000,NaN,NaN
2,03.d,03 std 10 1,Cal,NaN,3,1.0,C18_MRM_5dA Met 5MTA_Trp ISTD.m,0.795,549928.459,2851487.753,...,9.2884,10.0,92.9,2.831,1.145390e+07,2851487.753,4.0168,10.3151,10.0,103.2
3,04.d,04 std 25 1,Cal,NaN,3,2.0,C18_MRM_5dA Met 5MTA_Trp ISTD.m,0.795,776306.874,3255501.365,...,27.0404,25.0,108.2,2.819,1.747225e+07,3255501.365,5.3670,22.4425,25.0,89.8
4,05.d,05 std 75 1,Cal,NaN,3,3.0,C18_MRM_5dA Met 5MTA_Trp ISTD.m,0.795,1871660.733,2700113.710,...,73.5516,75.0,98.1,2.819,3.744101e+07,2700113.710,13.8665,81.1773,75.0,108.2


# Plot data